# 03 · Extra features (trends, composites, embeddings, interactions, visit trends)

Builds additional feature sets on top of `features2.parquet`. Only the first one (3.1) helped; the others are kept so you can see what was tried.

**Needs:** `features2.parquet` (notebook 02). **Writes:** `extras.parquet`, `features3.parquet` (the main set used later), `features4.parquet`, `features5.parquet`, `features6.parquet`. **Time:** about 6 minutes in total.

| Step | Idea | Result in our tests |
|---|---|---|
| 3.1 | Lab/vital trends, admission-vs-discharge windows, "how sick overall" scores | helped (about +0.001 AUC) |
| 3.2 | Diagnosis/medication SVD topics and code embeddings | slightly worse |
| 3.3 | 762 explicit interaction terms from 28 core variables | worse (see notebook 12) |
| 3.4 | Prior-visit acceleration and "which labs were ordered" | additive model 0.7522 vs 0.7524, no gain; the LightGBM comparison never finished |

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

### 3.1 Trend and composite features, then merge into `features3.parquet`

In [ ]:
import pandas as pd, numpy as np
D=DATA; rd=lambda n,**k: pd.read_csv(f'{D}/{n}.csv',**k)
ie=rd('index_encounters',parse_dates=['admit_dt','discharge_dt'])[['encounter_id','admit_dt','discharge_dt']]
# ---- labs (same cleaning as features.py) ----
lb=rd('lab_results',parse_dates=['collected_dt']).drop_duplicates(); lb['lab']=lb.lab_name.str.lower()
v=lb.value.values.copy(); u=lb.unit.values; nm=lb.lab.values
def mult(mask,f):
    global v; v=np.where(mask,v*f,v)
m=lambda lab,unit:(nm==lab)&(u==unit)
mult(m('albumin','g/L'),0.1); mult((nm=='albumin')&pd.isna(u)&(v>10),0.1); mult(m('creatinine','umol/L'),1/88.4); mult((nm=='creatinine')&pd.isna(u)&(v>25),1/88.4)
mult(m('glucose','mmol/L'),18.016); mult((nm=='glucose')&pd.isna(u)&(v<=30),18.016); mult(m('hemoglobin','g/L'),0.1); mult((nm=='hemoglobin')&pd.isna(u)&(v>30),0.1)
rng={'albumin':(1,6),'bnp':(5,40000),'bun':(2,250),'creatinine':(0.3,20),'glucose':(30,1000),'hba1c':(3,20),'hemoglobin':(3,22),'inr':(0.7,15),'lactate':(0.3,30),'potassium':(1.5,10),'sodium':(100,175),'troponin_i':(0,60),'wbc':(0.5,100)}
lo=np.array([rng[x][0] for x in nm]); hi=np.array([rng[x][1] for x in nm])
for _ in range(2): v=np.where(v>hi,v/10,v)
v=np.where((v<lo)&(v*10<=hi),v*10,v); v=np.where((v>hi)|(v<lo),np.nan,v); lb['v']=v
lb=lb.dropna(subset=['v']).merge(ie,on='encounter_id')
lb['t']=(lb.collected_dt-lb.admit_dt).dt.total_seconds()/86400; lb['hdc']=(lb.discharge_dt-lb.collected_dt).dt.total_seconds()/3600; lb['los']=(lb.discharge_dt-lb.admit_dt).dt.total_seconds()/86400
lb['tv']=lb.t*lb.v; lb['tt']=lb.t**2; lb['one']=1.0
g=lb.groupby(['encounter_id','lab'])
s=g[['one','t','v','tv','tt']].sum(); den=s.one*s.tt-s.t**2
E=pd.DataFrame({'slope':np.where(den>1e-6,(s.one*s.tv-s.t*s.v)/den,np.nan)},index=s.index)
E['std']=g.v.std()
w1=lb[lb.t<=1].groupby(['encounter_id','lab']).v.agg(adm24_mean='mean',adm24_max='max')
w2=lb[lb.hdc<=24].groupby(['encounter_id','lab']).v.agg(dc24_mean='mean',dc24_max='max',dc24_min='min')
pk=lb.loc[g.v.idxmax().dropna().astype(int).values].set_index(['encounter_id','lab']); E['peak_h_to_dc']=pk.hdc/(pk.los*24)
E=E.join(w1).join(w2); E['dc_minus_adm']=E.dc24_mean-E.adm24_mean
E=E.unstack('lab'); E.columns=[f'lx_{b}_{a}' for a,b in E.columns]
# composites: z-scored severity at discharge and at admission
last=lb.sort_values(['encounter_id','t']).groupby(['encounter_id','lab']).v.last().unstack(); first=lb.sort_values(['encounter_id','t']).groupby(['encounter_id','lab']).v.first().unstack()
mx=lb.groupby(['encounter_id','lab']).v.max().unstack(); mn=lb.groupby(['encounter_id','lab']).v.min().unstack()
def z(df,col,sign=1): 
    x=df[col]; return sign*(np.log(x) if col in('bnp','bun','creatinine','troponin_i','lactate','wbc','inr') else x).pipe(lambda s:(s-s.mean())/s.std()).clip(-4,4)
sev={'creatinine':1,'bun':1,'bnp':1,'albumin':-1,'hemoglobin':-1,'wbc':1,'sodium':-1,'lactate':1,'troponin_i':1,'glucose':1,'inr':1}
C=pd.DataFrame(index=last.index)
C['sev_last']=pd.concat([z(last,k,sg) for k,sg in sev.items() if k in last],axis=1).mean(axis=1)
C['sev_first']=pd.concat([z(first,k,sg) for k,sg in sev.items() if k in first],axis=1).mean(axis=1)
C['sev_worst']=pd.concat([z(mx if sg==1 else mn,k,sg) for k,sg in sev.items() if k in mx],axis=1).mean(axis=1)
C['kidney_sev']=(z(last,'creatinine')+z(last,'bun')).fillna(0)
crit=((last.sodium<135)|(last.sodium>145)).astype(float)+(last.potassium>5.0)+(last.potassium<3.5)+(last.creatinine>2)+(last.hemoglobin<10)+(last.bun>40)+(last.glucose>250)+(last.albumin<3)+(last.wbc>12)+(last.bnp>1000)
C['n_critical_last']=crit
del lb
# ---- vitals ----
vt=rd('vitals').drop_duplicates(); vt['t']=pd.to_datetime(vt.charttime,format='%m/%d/%Y %H:%M')
x=vt.value.values.copy(); ty=vt.vital_type.values
x=np.where((ty=='TEMP')&(x>50),(x-32)*5/9,x); x=np.where((ty=='HR')&((x>220)|(x<25)),np.nan,x); x=np.where((ty=='SPO2')&(x<60),np.nan,x)
vt['x']=x; vt=vt.dropna(subset=['x']).merge(ie,on='encounter_id'); vt['td']=(vt.t-vt.admit_dt).dt.total_seconds()/86400; vt['hdc']=(vt.discharge_dt-vt.t).dt.total_seconds()/3600
vt['tx']=vt.td*vt.x; vt['tt']=vt.td**2; vt['one']=1.0
g=vt.groupby(['encounter_id','vital_type']); s=g[['one','td','x','tx','tt']].sum(); den=s.one*s.tt-s.td**2
V=pd.DataFrame({'slope':np.where(den>1e-6,(s.one*s.tx-s.td*s.x)/den,np.nan)},index=s.index)
V=V.join(vt[vt.hdc<=12].groupby(['encounter_id','vital_type']).x.agg(dc12_mean='mean',dc12_min='min',dc12_max='max'))
V=V.join(vt[vt.hdc<=24].groupby(['encounter_id','vital_type']).x.agg(dc24_min='min',dc24_max='max',dc24_std='std'))
V=V.unstack('vital_type'); V.columns=[f'vx_{b}_{a}' for a,b in V.columns]
d12=vt[vt.hdc<=12]
inst=pd.DataFrame({'unstable_12h':((d12.vital_type=='SBP')&(d12.x<100)).groupby(d12.encounter_id).sum()+((d12.vital_type=='HR')&(d12.x>100)).groupby(d12.encounter_id).sum()+((d12.vital_type=='SPO2')&(d12.x<92)).groupby(d12.encounter_id).sum()+((d12.vital_type=='RR')&(d12.x>22)).groupby(d12.encounter_id).sum()+((d12.vital_type=='TEMP')&(d12.x>38)).groupby(d12.encounter_id).sum()})
inst['readings_12h']=d12.groupby('encounter_id').size(); inst['unstable_12h_frac']=inst.unstable_12h/inst.readings_12h
OUT=pd.concat([E,C,V,inst],axis=1); print(OUT.shape)
OUT.reset_index().to_parquet(f'{WORK}/extras.parquet')

In [ ]:
import pandas as pd
a = pd.read_parquet(f'{WORK}/features2.parquet'); b = pd.read_parquet(f'{WORK}/extras.parquet')
c3 = a.merge(b, on='encounter_id', how='left'); print(c3.shape); c3.to_parquet(f'{WORK}/features3.parquet')

### 3.2 Diagnosis / medication topics and embeddings (`features4.parquet`)

In [ ]:
import pandas as pd, numpy as np, scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
D=DATA; W=WORK; rd=lambda n,**k: pd.read_csv(f'{D}/{n}.csv',**k)
ids=rd('index_encounters')[['encounter_id']]; idx=pd.Index(ids.encounter_id); n=len(idx)
def norm(c):
    c=str(c).strip().upper().replace(' ','');  return c if ('.' in c or len(c)<=3) else c[:3]+'.'+c[3:]
dx=rd('diagnoses').drop_duplicates(); dx['code']=dx.icd10_code.map(norm)
codes=sorted(dx.code.unique()); ci={c:i for i,c in enumerate(codes)}
B=sp.csr_matrix((np.ones(len(dx)),(idx.get_indexer(dx.encounter_id),dx.code.map(ci))),shape=(n,len(codes))); B.data[:]=1
md=rd('medications'); md['d']=md.drug_name.str.strip().str.lower(); md=md.drop_duplicates(['encounter_id','d']); dn=sorted(md.d.unique())
M=sp.csr_matrix((np.ones(len(md)),(idx.get_indexer(md.encounter_id),md.d.map({d:i for i,d in enumerate(dn)}))),shape=(n,len(dn)))
out={}
# 1) SVD topics (dx, meds, joint)
for name,Mx,k in [('dxsvd',B,12),('medsvd',M,8),('joint',sp.hstack([B,M]).tocsr(),12)]:
    Z=TruncatedSVD(k,random_state=0).fit_transform(Mx); 
    for j in range(k): out[f'{name}_{j}']=Z[:,j]
# 2) code embeddings: PPMI of co-occurrence + SVD (word2vec-equivalent), averaged per stay
C=(B.T@B).toarray(); np.fill_diagonal(C,0); tot=C.sum(); pc=C.sum(1,keepdims=True)/tot; pmi=np.log((C/tot+1e-12)/(pc@pc.T+1e-12)); ppmi=np.maximum(pmi,0)
U,s,_=np.linalg.svd(ppmi); E=U[:,:10]*np.sqrt(s[:10]); cnt=np.asarray(B.sum(1)).ravel().clip(1); Em=(B@E)/cnt[:,None]
for j in range(10): out[f'dxemb_{j}']=Em[:,j]
prim=dx.sort_values(['encounter_id','is_primary','seq'],ascending=[True,False,True]).drop_duplicates('encounter_id').set_index('encounter_id').code
Ep=E[prim.reindex(idx).map(ci).fillna(0).astype(int).values]
for j in range(10): out[f'primemb_{j}']=Ep[:,j]
# 3) medication-class intensity topics
out['n_med_drugs_unique']=np.asarray((M>0).sum(1)).ravel()
F=pd.DataFrame(out); F.insert(0,'encounter_id',idx.values); F.to_parquet(f'{W}/feat4_extra.parquet'); print(F.shape)
a=pd.read_parquet(f'{W}/features3.parquet'); b=a.merge(F,on='encounter_id',how='left'); b.to_parquet(f'{W}/features4.parquet'); print(b.shape)

### 3.3 Explicit interaction terms (`features5.parquet`)
The columns are chosen from medical reasoning only (never from the target), so there is no selection leakage.

In [ ]:
import pandas as pd, numpy as np
W=WORK
X=pd.read_parquet(f'{W}/features3.parquet')
# 25 clinically central variables, chosen by domain reasoning only (NOT by target) -> no selection leakage
core=['age','los_days','lab_creatinine_last','lab_creatinine_max','lab_bun_last','egfr_last','lab_albumin_last',
      'lab_hemoglobin_last','lab_sodium_last','lab_bnp_max','lab_wbc_last','lab_glucose_last','n_dx','n_chronic',
      'charlson_like','lace','pu_total','pu_acute_365','pu_ED_180','deprivation_index','sev_last','sev_worst',
      'n_critical_last','dp_followup_scheduled_days','n_med','med_cont_frac','vit_SPO2_min','unstable_12h_frac']
core=[c for c in core if c in X.columns]; print('core vars:',len(core))
Z=X[core].astype('float64')
# rank-normalise each to [0,1] so products are stable and comparable (monotone, target-free)
R=Z.rank(pct=True).fillna(0.5)
out={}
for i in range(len(core)):
    for j in range(i+1,len(core)):
        a,b=core[i],core[j]
        out[f'ix_{a}__X__{b}']=(R[a]*R[b]).astype('float32')          # joint high/low  = 2-way interaction term
        out[f'ix_{a}__D__{b}']=(R[a]-R[b]).astype('float32')          # divergence between the two
F=pd.DataFrame(out); F.insert(0,'encounter_id',X.encounter_id.values)
# a few clinically meaningful raw ratios too
F['rx_bun_cr']=(X.lab_bun_last/X.lab_creatinine_last).astype('float32')
F['rx_cr_per_age']=(X.lab_creatinine_last/X.age.clip(lower=18)).astype('float32')
F['rx_sev_x_nofollowup']=(X.sev_last.fillna(0)*X.dp_followup_missing.fillna(1)).astype('float32')
F['rx_meds_per_dx']=(X.n_med/X.n_dx.clip(lower=1)).astype('float32')
F['rx_acute_per_chronic']=(X.pu_acute_365/X.n_chronic.clip(lower=1)).astype('float32')
print('interaction features:',F.shape)
F.to_parquet(f'{W}/feat5_ix.parquet')
B=X.merge(F,on='encounter_id',how='left'); B.to_parquet(f'{W}/features5.parquet'); print('features5:',B.shape)

### 3.4 Visit acceleration and lab-ordering features (`features6.parquet`)

In [ ]:
import pandas as pd, numpy as np
D=DATA; W=WORK
ie=pd.read_csv(f'{D}/index_encounters.csv',parse_dates=['admit_dt','discharge_dt'])[['encounter_id','patient_id','admit_dt','hospital_id']]
pu=pd.read_csv(f'{D}/prior_utilization.csv',parse_dates=['visit_dt']).drop_duplicates().merge(ie,on='patient_id',suffixes=('','_idx'))
pu['d']=(pu.admit_dt-pu.visit_dt).dt.days
F=pd.DataFrame(index=ie.encounter_id)
# visit counts in 4 half-year bins (bin0 = most recent) and the trend across them
pu['bin']=np.minimum(pu.d//183,3)
for tp,s in [('all',pu),('acute',pu[pu.visit_type.isin(['ED','IP'])]),('OP',pu[pu.visit_type=='OP'])]:
    c=s.groupby(['encounter_id','bin']).size().unstack(fill_value=0).reindex(columns=range(4),fill_value=0).reindex(F.index,fill_value=0)
    for k in range(4): F[f'pa_{tp}_bin{k}']=c[k]
    x=np.array([0,1,2,3]); xc=x-x.mean()
    F[f'pa_{tp}_trend']=-(c.values@xc)/(xc**2).sum()             # >0 = visits increasing toward admission
    F[f'pa_{tp}_accel']=(c[0]+0.5)/((c[1]+c[2]+c[3])/3+0.5)      # recent half-year vs older average
    r90=s[s.d<=90].groupby('encounter_id').size().reindex(F.index,fill_value=0)
    F[f'pa_{tp}_r90_vs_rest']=(r90/90+0.002)/((s.groupby('encounter_id').size().reindex(F.index,fill_value=0)-r90)/640+0.002)
# gap patterns
s=pu.sort_values(['encounter_id','visit_dt']); s['gap']=s.groupby('encounter_id').visit_dt.diff().dt.days
g=s.groupby('encounter_id').gap
F['pa_gap_median']=g.median(); F['pa_gap_last']=s.groupby('encounter_id').gap.last(); F['pa_gap_min']=g.min()
F['pa_gap_last_vs_med']=F.pa_gap_last/(F.pa_gap_median+1)
F['pa_days_since_any']=pu.groupby('encounter_id').d.min()
F['pa_acute_last_type_IP']=(s[s.visit_type.isin(['ED','IP'])].groupby('encounter_id').visit_type.last()=='IP').astype(float)
F['pa_n_hosp_other']=pu[pu.hospital_id!=pu.hospital_id_idx].groupby('encounter_id').hospital_id.nunique()
# which lab tests were ordered (a signal of what the clinician suspected), using raw rows
lb=pd.read_csv(f'{D}/lab_results.csv',usecols=['encounter_id','lab_name','collected_dt'],parse_dates=['collected_dt']).drop_duplicates()
lb['lab']=lb.lab_name.str.lower(); lb=lb.merge(ie[['encounter_id','admit_dt']],on='encounter_id')
lb['h']=(lb.collected_dt-lb.admit_dt).dt.total_seconds()/3600
o=lb.groupby(['encounter_id','lab']).size().unstack(fill_value=0).reindex(F.index,fill_value=0)
for c in o.columns: F[f'ord_{c}']=(o[c]>0).astype(int)
late=lb[lb.h>48].groupby(['encounter_id','lab']).size().unstack(fill_value=0).reindex(F.index,fill_value=0)
for c in ['troponin_i','lactate','bnp','inr']:
    if c in late: F[f'ord_late_{c}']=(late[c]>0).astype(int)
F['ord_acute_panel']=F[[f'ord_{c}' for c in ['troponin_i','lactate','bnp','inr'] if f'ord_{c}' in F]].sum(1)
F['ord_n_types']=(o>0).sum(1)
F=F.reset_index(); print(F.shape); F.to_parquet(f'{W}/feat6_extra.parquet')
B=pd.read_parquet(f'{W}/features3.parquet').merge(F,on='encounter_id',how='left'); B.to_parquet(f'{W}/features6.parquet'); print('features6',B.shape)